# Experimento — KAN, fase 5: placar comum e teste final

Todos os candidatos escolhidos nas fases anteriores são comparados no protocolo comum `comum.medir_placar`: 3×5 dobras, treino interno para escolher passos e comparações pareadas por cliente com IC95. A referência KAN e o campeão são registrados no placar do projeto; nenhum resultado de teste participa da adoção.

**Execute as células na ordem.** A célula final abre o teste uma única vez, depois de registrar e congelar o campeão. Não altere a configuração com base no resultado do teste.

## 0. Preparação

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

CSV_PATH = os.environ.get("CHURN_CSV", "")
if CSV_PATH:
    os.environ["CHURN_CSV"] = CSV_PATH
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório churn_project.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import comum
from experimentos.kan import modelo as K

RESULTADOS = Path(comum.RESULTADOS)
PASTA_KAN = RESULTADOS / "kan"
PASTA_KAN.mkdir(parents=True, exist_ok=True)
print(f"Repositório: {ROOT}")
print(f"Dispositivo: {K.DEVICE}")
print(f"CSV: {comum.CAMINHO_CSV}")

In [ ]:
ARQUIVO = PASTA_KAN / "kan_fase4.json"
if not ARQUIVO.is_file():
    raise FileNotFoundError(f"Execute antes a fase anterior: {ARQUIVO}")
REGISTRO = json.loads(ARQUIVO.read_text(encoding="utf-8"))
CAMPEAO = dict(REGISTRO["config"])
CAMPEAO["hidden"] = tuple(CAMPEAO["hidden"])
FEATURES = tuple(REGISTRO.get("features", []))
OVER = REGISTRO.get("over", True)
print("Configuração recebida:", CAMPEAO)
print("Features:", FEATURES, "| oversampling:", OVER)

N_SEMENTES_TESTE = 5
ROTULO_CAMPEAO = "kan_final"

## 1. Mesma régua para referência e campeão

In [ ]:
X_train, _, y_train, _ = comum.dividir(comum.carregar_df())
assinatura = comum.assinatura_treino(X_train)

medida_ref = K.avaliar_placar(K.KAN_PADRAO)
comum.registrar_placar(RESULTADOS, "kan", "referencia", K.KAN_PADRAO,
                       medida_ref, y_train, assinatura)

medida_final = K.avaliar_placar(CAMPEAO, FEATURES, OVER)
config_registro = {**CAMPEAO, "hidden": list(CAMPEAO["hidden"]),
                   "features": list(FEATURES), "over": OVER}
comum.registrar_placar(RESULTADOS, "kan", ROTULO_CAMPEAO, config_registro,
                       medida_final, y_train, assinatura,
                       contra={"vs_referencia": ("kan", "referencia")})
comum.adotar_placar(RESULTADOS, "kan", ROTULO_CAMPEAO,
                    "Maior KS no placar comum, escolhido antes de abrir o teste.")
print(comum.tabela_placar(RESULTADOS).query("modelo == 'kan'"))

## 2. Congelar os passos e abrir o teste uma única vez

In [ ]:
# A configuração e o número de passos foram definidos sem consultar o teste.
X_train, X_test, y_train, y_test = comum.dividir(comum.carregar_df())
passos_finais = int(medida_final["steps_mediana"])
p_teste_ref = np.asarray([
    K.prever_teste(K.KAN_PADRAO, seed=400 + i)
    for i in range(N_SEMENTES_TESTE)
])
registro_ref = comum.registrar_teste_placar(
    RESULTADOS, "kan", "referencia", y_test.to_numpy(), p_teste_ref
)
p_teste = np.asarray([
    K.prever_teste(CAMPEAO, FEATURES, OVER, passos_finais, seed=500 + i)
    for i in range(N_SEMENTES_TESTE)
])
registro_teste = comum.registrar_teste_placar(
    RESULTADOS, "kan", ROTULO_CAMPEAO, y_test.to_numpy(), p_teste
)
p = p_teste.mean(axis=0)
limiar = comum.limiar_ks(y_train.to_numpy(), medida_final["oof"].mean(axis=0))
metricas_teste = comum.metricas(y_test.to_numpy(), p, limiar)
print("KS da referência no teste:", registro_ref["ks_teste"])
print("Métricas do campeão no teste (somente avaliação final):", metricas_teste)
print("KS IC95:", registro_teste["ks_teste_ic95"])

## 3. Curvas finais: ROC, precision-recall e KS

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

y = y_test.to_numpy()
fpr, tpr, limiares_roc = roc_curve(y, p)
precisao, recall, _ = precision_recall_curve(y, p)
ks_curva = tpr - fpr
i_ks = int(np.argmax(ks_curva))
prevalencia = float(y.mean())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(fpr, tpr, label=f"KAN (AUROC={roc_auc_score(y, p):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC — teste", xlabel="taxa de falsos positivos", ylabel="taxa de verdadeiros positivos")
axes[0].legend()
axes[1].plot(recall, precisao, label=f"AP={average_precision_score(y, p):.3f}")
axes[1].axhline(prevalencia, linestyle="--", color="gray", label=f"prevalência={prevalencia:.1%}")
axes[1].set(title="Precision-recall — teste", xlabel="recall", ylabel="precision")
axes[1].legend()
for classe, nome in ((0, "ficam"), (1, "cancelam")):
    escores = np.sort(p[y == classe])
    cdf = np.arange(1, len(escores) + 1) / len(escores)
    axes[2].plot(escores, cdf, label=nome)
axes[2].axvline(limiares_roc[i_ks], color="black", linestyle="--",
                label=f"KS={ks_curva[i_ks]:.3f}")
axes[2].set(title="KS — teste", xlabel="probabilidade prevista de churn", ylabel="fração acumulada")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()